# MS in Applied Data Science — RAG Baseline
ADSP 32027: Generative AI Principles — Course Project 1

The simplest end-to-end RAG pipeline over the [MS in Applied Data Science website](https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/):

1. **Scrape** the program pages (`scraper.py`) and save them to `data/`
2. **Load** each page as a LangChain `Document` (URL + title as metadata)
3. **Chunk** with `RecursiveCharacterTextSplitter`
4. **Embed** with OpenAI `text-embedding-3-small` and store in **Chroma**
5. **Retrieve + generate** answers with OpenAI `gpt-4o-mini`, citing source URLs

Then one improvement (**heading-aware chunks**) and a **RAGAS evaluation** of both pipelines on a shared test set
(`eval/testset.json`, scored by `rag_eval.py`). Every other approach should be scored the same way so results are
comparable.

## Setup
Requires `OPENAI_API_KEY` in `Midterm/.env`.

In [1]:
import json
import os
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=DeprecationWarning)
load_dotenv()  # reads OPENAI_API_KEY from .env
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found -- add it to Midterm/.env"

import scraper  # Midterm/scraper.py

# ── Config ────────────────────────────────────────────────────────────────────
RUN_SCRAPER = True     # set to False to reuse data/pages.jsonl from the last crawl (~2 min saved)
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4
EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"
CHROMA_DIR = "chroma_db"
COLLECTION = "msads_baseline"

## Stage 1: Scrape and save
Crawls the program site (main page + sublinks, faculty profiles, MS-ADS news posts) and writes
`data/raw_pages/*.txt` and `data/pages.jsonl`. See `scraper.py` for crawl scope and cleaning rules.

In [2]:
if RUN_SCRAPER:
    pages, failures = scraper.crawl()
    scraper.save_pages(pages)
    scraper.print_summary(pages, failures)
else:
    print(f"Skipping crawl -- using existing {scraper.JSONL_PATH}")

  [  1] depth=0 chars=  2183  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/


  [  2] depth=1 chars=  6651  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/


  [  3] depth=1 chars= 36559  https://datascience.uchicago.edu/education/masters-programs/in-person-program/


  [  4] depth=1 chars= 34077  https://datascience.uchicago.edu/education/masters-programs/online-program/


  [  5] depth=1 chars= 58894  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/instructors-staff/


  [  6] depth=1 chars=  4453  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/


... (65 outputs left out)

  [ 72] depth=2 chars=   728  https://datascience.uchicago.edu/people/samantha-widemon/


  [ 73] depth=2 chars=   784  https://datascience.uchicago.edu/people/ben-wiebers/


  [ 74] depth=2 chars=  6694  https://datascience.uchicago.edu/news/the-tools-languages-and-technologies-ive-actually-used-in-ms-ads/


  [ 75] depth=2 chars=  5195  https://datascience.uchicago.edu/news/should-you-choose-ms-in-data-science-or-ms-in-applied-data-science-at-uchicago/

Scraped 75 pages | 303,971 total chars
Saved to: C:\Users\alber\OneDrive\Escritorio\Documentos\UChicago\Gen AI\Midterm\data\raw_pages  and  C:\Users\alber\OneDrive\Escritorio\Documentos\UChicago\Gen AI\Midterm\data\pages.jsonl

1 skipped/failed URL(s):
  - https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/tuition-fees-aid/: redirects to already-saved https://datascience.uchicago.edu/education/tuition-fees-aid/


## Stage 2: Load documents
One `Document` per page. Loading from `pages.jsonl` (rather than the `.txt` files) gives every chunk a clean
`source` URL and `title` for citations.

In [3]:
from langchain_core.documents import Document


def load_documents(jsonl_path=scraper.JSONL_PATH):
    docs = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            page = json.loads(line)
            docs.append(Document(
                page_content=page["text"],
                metadata={"source": page["url"], "title": page["title"]},
            ))
    return docs


docs = load_documents()
print(f"Loaded {len(docs)} documents | {sum(len(d.page_content) for d in docs):,} chars")

Loaded 75 documents | 303,971 chars


## Stage 3: Chunk
The scraper puts a blank line before every heading (sections, FAQ questions, course names), so the splitter's
default `"\n\n"` separator already breaks mostly at section boundaries.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)

lengths = [len(c.page_content) for c in chunks]
print(f"Created {len(chunks)} chunks | avg {sum(lengths) / len(lengths):.0f} chars | max {max(lengths)} chars")

sample = next(c for c in chunks if c.metadata["source"].endswith("/faqs/"))
print(f"\nSample chunk from {sample.metadata['source']}:\n{'-' * 70}\n{sample.page_content}")

Created 631 chunks | avg 504 chars | max 799 chars

Sample chunk from https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/:
----------------------------------------------------------------------
# FAQs

# Master’s in Applied Data Science FAQs
Learn more about what makes our program unique.
David Uminsky, PhD – UChicago Data Science Institute, Executive Director

### Application Process

#### When will I receive my Master's in Applied Data Science admission decision?
Admissions decisions are typically released 1-2 months after each application deadline. Only completed applications are reviewed. Please refer to the How to Apply page for guidelines.

#### If I finish my Master's in Applied Data Science application before the deadline, will I receive my decision early?
No, admissions decisions for the in-person program are typically released 1-2 months after each application deadline. Your application must be complete to be considered for review.


## Stage 4: Embed and store in Chroma
The collection is rebuilt from scratch on every run so re-running the notebook never duplicates vectors.

In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)



def build_store(chunks, collection_name: str) -> Chroma:
    """(Re)build a persistent Chroma collection from scratch so re-runs never duplicate vectors."""
    Chroma(collection_name=collection_name, embedding_function=embeddings,
           persist_directory=CHROMA_DIR).delete_collection()
    store = Chroma.from_documents(chunks, embeddings, collection_name=collection_name,
                                  persist_directory=CHROMA_DIR)
    print(f"[{collection_name}] vector store contains {store._collection.count()} vectors")
    return store


vectorstore = build_store(chunks, COLLECTION)
print(f"Embedding dimension: {len(embeddings.embed_query('test'))}")

[msads_baseline] vector store contains 631 vectors


Embedding dimension: 1536


## Stage 5: Retrieval + generation
Retrieve the top-k chunks, then ask the LLM to answer **only** from that context, cite the source URLs,
and fall back to a fixed message when the answer isn't in the context.

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

FALLBACK = "I don't have that information in the MS in Applied Data Science website content."

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for the University of Chicago's MS in Applied Data Science program.\n"
     "Answer the user's question using ONLY the context below.\n"
     "- Be concise and specific.\n"
     "- End your answer with a 'Sources:' line listing the URL(s) you used.\n"
     "- If the context does not contain the answer, reply exactly: \"{fallback}\"\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
]).partial(fallback=FALLBACK)

generate = prompt | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{i}] {d.metadata['title']} ({d.metadata['source']})\n{d.page_content}"
        for i, d in enumerate(docs, 1)
    )


def ask(question: str, retriever=retriever, verbose: bool = True) -> dict:
    """Retrieve context for the question, generate a grounded answer, and show what was retrieved."""
    retrieved = retriever.invoke(question)
    answer = generate.invoke({"context": format_docs(retrieved), "question": question})
    if verbose:
        print(f"Q: {question}\n\nA: {answer}\n\nRetrieved chunks:")
        for i, d in enumerate(retrieved, 1):
            print(f"  [{i}] {d.metadata['title']} -- {d.metadata['source']}")
        print("=" * 70)
    return {"question": question, "answer": answer,
            "contexts": [d.page_content for d in retrieved],  # what RAGAS judges retrieval/faithfulness against
            "sources": [d.metadata["source"] for d in retrieved]}

## Stage 6: Try it (baseline)
The first three questions are the example interactions from the project brief; the last one is out of scope
and should trigger the fallback message.

In [7]:
TEST_QUESTIONS = [
    # examples from the project brief
    "What are the core courses in the MS in Applied Data Science program?",
    "What are the admission requirements for the MS in Applied Data Science program?",
    "Can you provide information about the capstone project?",
    # additional coverage
    "How much is tuition for the program?",
    "How does the online program differ from the in-person program?",
    "Is the program STEM/OPT eligible for international students?",
    "When are the application deadlines?",
    # out of scope -> should return the fallback
    "What is the best pizza place in Chicago?",
]

baseline_results = [ask(q) for q in TEST_QUESTIONS]

Q: What are the core courses in the MS in Applied Data Science program?

A: I don't have that information in the MS in Applied Data Science website content.

Retrieved chunks:
  [1] Course Progressions -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions/
  [2] Online Program -- https://datascience.uchicago.edu/education/masters-programs/online-program/
  [3] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
  [4] Should You Choose MS in Data Science or MS in Applied Data Science at UChicago? -- https://datascience.uchicago.edu/news/should-you-choose-ms-in-data-science-or-ms-in-applied-data-science-at-uchicago/


Q: What are the admission requirements for the MS in Applied Data Science program?

A: The admission requirements for the MS in Applied Data Science program include some prior experience with Python or R. Additionally, applicants must submit a completed application, which will be reviewed by the committee. The application portal may close early if the cohort is filled. 

Sources: 
- https://datascience.uchicago.edu/news/applying-to-the-ms-in-applied-data-science-program-heres-what-we-look-for/
- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/

Retrieved chunks:
  [1] Applying to the MS in Applied Data Science Program? Here’s What We Look For -- https://datascience.uchicago.edu/news/applying-to-the-ms-in-applied-data-science-program-heres-what-we-look-for/
  [2] How to Apply -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/
  [3] Course Progressions -- https://datascience.uchicago.

Q: Can you provide information about the capstone project?

A: The Capstone Project in the MS in Applied Data Science program is a required component that is completed over two quarters. It involves research design, implementation, and writing. Full-time students typically start their capstone project in their third quarter, while part-time students generally begin in their fifth quarter. 

In the second quarter, the capstone class focuses on providing students with maximum flexibility to work with their Capstone advisors on concluding the execution of the analytic methodology and any client/sponsor deliverables. It also supports students in curating and delivering key project communications, including a formal research paper, a formal business presentation, and a live presentation during the Capstone Showcase with a Q&A session.

Sources: 
- https://datascience.uchicago.edu/education/masters-programs/in-person-program/
- https://datascience.uchicago.edu/education/masters-programs/onli

Q: How much is tuition for the program?

A: Tuition for the MS in Applied Data Science program is $6,384 per course, totaling $76,608 for 12 courses or $114,912 for 18 courses. Additionally, there is a non-refundable program enrollment deposit of $1,500, which is credited toward your first quarter’s tuition balance. 

Sources: https://datascience.uchicago.edu/education/tuition-fees-aid/

Retrieved chunks:
  [1] Tuition, Fees, & Aid -- https://datascience.uchicago.edu/education/tuition-fees-aid/
  [2] Tuition, Fees, & Aid -- https://datascience.uchicago.edu/education/tuition-fees-aid/
  [3] How to Apply -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/
  [4] Tuition, Fees, & Aid -- https://datascience.uchicago.edu/education/tuition-fees-aid/


Q: How does the online program differ from the in-person program?

A: The online program differs from the in-person program primarily in the mode of delivery. The in-person program is conducted in a classroom environment in Chicago, providing access to on-campus student services and amenities. In contrast, the online program allows for flexible learning, with classes held both synchronously and asynchronously, typically during evenings and weekends. However, both programs share the same curriculum standards and are overseen by the same faculty committee. 

Sources:
- https://datascience.uchicago.edu/education/masters-programs/in-person-program/
- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/

Retrieved chunks:
  [1] In-Person Program -- https://datascience.uchicago.edu/education/masters-programs/in-person-program/
  [2] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
  [3] Course Progres

Q: Is the program STEM/OPT eligible for international students?

A: Yes, the full-time, In-Person Master’s in Applied Data Science program is listed as a STEM-designated degree, making it eligible for the STEM OPT extension for international students. However, approval of STEM OPT is at the discretion of U.S. Citizenship & Immigration Services. 

Sources: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/

Retrieved chunks:
  [1] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
  [2] In-Person Program -- https://datascience.uchicago.edu/education/masters-programs/in-person-program/
  [3] International Students -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/international-students/
  [4] Online Program -- https://datascience.uchicago.edu/education/masters-programs/online-program/


Q: When are the application deadlines?

A: The application deadlines for the MS in Applied Data Science program are as follows:

**In-Person:**
- November 9, 2026 – First Priority Application Deadline
- December 7, 2026 – Second Priority Application Deadline
- January 26, 2027 – International Application Deadline (requiring visa sponsorship from UChicago)
- March 4, 2027 – Third Priority Application Deadline
- May 6, 2027 – Fourth Priority Application Deadline
- June 23, 2027 – Final Application Deadline

**Online:**
- June 23, 2027 – Final Application Deadline

Sources: 
https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/
https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/

Retrieved chunks:
  [1] Events & Deadlines -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/
  [2] How to Apply -- https://datascience.uchicago.edu/educatio

Q: What is the best pizza place in Chicago?

A: I don't have that information in the MS in Applied Data Science website content.

Retrieved chunks:
  [1] Explore the MS-ADS Campus -- https://datascience.uchicago.edu/explore-the-ms-ads-campus/
  [2] Should You Choose MS in Data Science or MS in Applied Data Science at UChicago? -- https://datascience.uchicago.edu/news/should-you-choose-ms-in-data-science-or-ms-in-applied-data-science-at-uchicago/
  [3] DuJuan Smith, PhD -- https://datascience.uchicago.edu/people/dujuan-smith/
  [4] Briana Allen, MEd -- https://datascience.uchicago.edu/people/briana-allen/


---
# Improvement 1: Heading-aware chunks
**Problem found in the baseline:** *"What are the core courses?"* returned the fallback. The in-person page lists
each core course under a `### Core Courses` tab, but after plain character splitting a course's chunk contains only
its description — not the words "Core Courses" — so the right chunks are never retrieved. Worse, course
descriptions (~1,000 chars) are longer than a chunk, so the splitter often cuts right after the course title and
leaves the title as its own near-empty chunk. *"What are the admission requirements?"* has the same shape: the
requirements are separate `##` sections of the How to Apply page, and no single chunk lists them.

**Fix:**
1. **Split at every heading level** (`#`–`######`) with `MarkdownHeaderTextSplitter`, so each course, FAQ question,
   or application requirement is its own section. Long sections are split further with the same
   `RecursiveCharacterTextSplitter` (800 / 150).
2. **Breadcrumb on every chunk:** `page title > section > subsection > item` is prepended to each piece, so the
   embedding and the LLM always know what the text belongs to (no more orphaned headings).
3. **Outline chunks:** for every heading with 2+ child headings, add one chunk listing those children
   (e.g. *Core Courses → Time Series…, Statistical Models…, Machine Learning I…*). These are built from the page
   structure, not generated by an LLM, and directly answer "list"-type questions.

Everything else (embedding model, k, prompt, LLM) is unchanged, so differences come from chunking alone.

In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

HEADERS = [("#" * n, f"h{n}") for n in range(1, 7)]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS, strip_headers=True)


def heading_path(doc, section):
    """[page title, h1, h2, ...] for a section, without repeats (the h1 usually equals the page title)."""
    path = [doc.metadata["title"]] + [section.metadata[name] for _, name in HEADERS if name in section.metadata]
    return [p for i, p in enumerate(path) if p and p not in path[:i]]


def heading_aware_chunks(docs):
    chunks = []
    for doc in docs:
        children = {}  # heading path -> its child headings, in page order
        for section in header_splitter.split_text(doc.page_content):
            path = heading_path(doc, section)
            for depth in range(1, len(path)):
                siblings = children.setdefault(tuple(path[:depth]), [])
                if path[depth] not in siblings:
                    siblings.append(path[depth])

            # content chunks: breadcrumb + section text
            breadcrumb = " > ".join(path)
            meta = {**doc.metadata, "section": breadcrumb, "chunk_type": "content"}
            for piece in splitter.split_text(section.page_content):
                chunks.append(Document(page_content=f"{breadcrumb}\n\n{piece}", metadata=meta))

        # outline chunks: breadcrumb + list of child headings
        for parent, kids in children.items():
            if len(kids) < 2:
                continue
            breadcrumb = " > ".join(parent)
            meta = {**doc.metadata, "section": breadcrumb, "chunk_type": "outline"}
            outline = "\n".join(f"- {k}" for k in kids)
            for piece in splitter.split_text(f"Contents of '{parent[-1]}':\n{outline}"):
                chunks.append(Document(page_content=f"{breadcrumb}\n\n{piece}", metadata=meta))
    return chunks


chunks_v2 = heading_aware_chunks(docs)
lengths = [len(c.page_content) for c in chunks_v2]
n_outline = sum(c.metadata["chunk_type"] == "outline" for c in chunks_v2)
print(f"Created {len(chunks_v2)} chunks ({n_outline} outline) | avg {sum(lengths) / len(lengths):.0f} chars | "
      f"max {max(lengths)} chars")

for label, chunk_type in [("content", "content"), ("outline", "outline")]:
    sample = next(c for c in chunks_v2
                  if "Core Courses" in c.metadata["section"] and c.metadata["chunk_type"] == chunk_type)
    print(f"\nSample {label} chunk:\n{'-' * 70}\n{sample.page_content[:500]}")

Created 718 chunks (60 outline) | avg 518 chars | max 978 chars

Sample content chunk:
----------------------------------------------------------------------
In-Person Program > Comparison Chart > Core Courses > Time Series Analysis and Forecasting

Time Series Analysis is a science as well as the art of making rational predictions based on previous records. It is widely used in various fields in today’s business settings. For example, airline companies employ time series to predict traffic volume and schedule flights; financial agencies measure market risk via stock price series; marketing analysts study the impact of a newly proposed advertisement 

Sample outline chunk:
----------------------------------------------------------------------
In-Person Program > Comparison Chart > Core Courses

Contents of 'Core Courses':
- Time Series Analysis and Forecasting
- Statistical Models for Data Science
- Machine Learning I
- Machine Learning II
- Cloud-Native Data Engineering
- Leadership a

In [9]:
vectorstore_v2 = build_store(chunks_v2, "msads_headings")
retriever_v2 = vectorstore_v2.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

headings_results = [ask(q, retriever=retriever_v2) for q in TEST_QUESTIONS]

[msads_headings] vector store contains 718 vectors


Q: What are the core courses in the MS in Applied Data Science program?

A: The core courses in the MS in Applied Data Science program are:

1. Time Series Analysis and Forecasting
2. Statistical Models for Data Science
3. Machine Learning I
4. Machine Learning II
5. Cloud-Native Data Engineering
6. Leadership and Consulting for Data Science
7. Data Science Capstone Project

Sources: https://datascience.uchicago.edu/education/masters-programs/online-program/

Retrieved chunks:
  [1] Course Progressions -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions/
  [2] Online Program -- https://datascience.uchicago.edu/education/masters-programs/online-program/
  [3] Applying to the MS in Applied Data Science Program? Here’s What We Look For -- https://datascience.uchicago.edu/news/applying-to-the-ms-in-applied-data-science-program-heres-what-we-look-for/
  [4] Online Program -- https://datascience.uchicago.edu/education/masters-programs

Q: What are the admission requirements for the MS in Applied Data Science program?

A: The admission requirements for the MS in Applied Data Science program include:

- Next Application Deadlines
- Letters of Recommendation
- Candidate Statement
- Resume/CV
- Programming Supplement
- Virtual Portfolio
- Transcripts from all previous colleges and universities attended
- Application Fee
- English Language Requirement (for international students)

Sources: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/

Retrieved chunks:
  [1] How to Apply -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/
  [2] Applying to the MS in Applied Data Science Program? Here’s What We Look For -- https://datascience.uchicago.edu/news/applying-to-the-ms-in-applied-data-science-program-heres-what-we-look-for/
  [3] Applying to the MS in Applied Data Science Program? Here’s What We Look For -- https://datascie

Q: Can you provide information about the capstone project?

A: The capstone project in the MS in Applied Data Science program allows students to work on real-world problems for actual clients, enhancing their collaboration skills and providing networking opportunities. Faculty-mentored teams contribute to various sectors, including finance, entertainment, and automotive. In the second quarter, students focus on executing their analytic methodology and preparing deliverables, which include a formal research paper, a business presentation, and a live presentation during the Capstone Showcase.

Sources: 
- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/
- https://datascience.uchicago.edu/education/masters-programs/in-person-program/
- https://datascience.uchicago.edu/education/masters-programs/online-program/

Retrieved chunks:
  [1] Capstone Projects -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data

Q: How much is tuition for the program?

A: Tuition for the MS in Applied Data Science program is $6,384 per course, totaling $76,608 for 12 courses or $114,912 for 18 courses. Additionally, there is a non-refundable program enrollment deposit of $1,500, which is credited toward your first quarter’s tuition balance. 

Sources: https://datascience.uchicago.edu/education/tuition-fees-aid/

Retrieved chunks:
  [1] Tuition, Fees, & Aid -- https://datascience.uchicago.edu/education/tuition-fees-aid/
  [2] Tuition, Fees, & Aid -- https://datascience.uchicago.edu/education/tuition-fees-aid/
  [3] Tuition, Fees, & Aid -- https://datascience.uchicago.edu/education/tuition-fees-aid/
  [4] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/


Q: How does the online program differ from the in-person program?

A: The online program differs from the in-person program primarily in the mode of delivery. The online program allows for flexibility and is designed for those who may need to balance work and life commitments, while the in-person program is ideal for those who prefer a traditional classroom environment and access to on-campus services. Both programs maintain the same academic rigor and are overseen by the same faculty curriculum committee. 

Sources: 
- https://datascience.uchicago.edu/education/masters-programs/online-program/
- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/

Retrieved chunks:
  [1] Online Program -- https://datascience.uchicago.edu/education/masters-programs/online-program/
  [2] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
  [3] In-Person Program -- https://datascience.uchicago.edu/education/masters

Q: Is the program STEM/OPT eligible for international students?

A: Yes, the full-time, In-Person Master’s in Applied Data Science program is listed as a STEM-designated degree, allowing eligible international students to apply for the STEM OPT extension. However, approval is at the discretion of U.S. Citizenship & Immigration Services. 

Sources: 
https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/

Retrieved chunks:
  [1] In-Person Program -- https://datascience.uchicago.edu/education/masters-programs/in-person-program/
  [2] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
  [3] FAQs -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/
  [4] International Students -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/international-students/


Q: When are the application deadlines?

A: The application deadlines for the MS in Applied Data Science program are as follows:

**In-Person:**
- November 9, 2026 – First Priority Application Deadline
- December 7, 2026 – Second Priority Application Deadline
- January 26, 2027 – International Application Deadline (requiring visa sponsorship from UChicago)
- March 4, 2027 – Third Priority Application Deadline
- May 6, 2027 – Fourth Priority Application Deadline
- June 23, 2027 – Final Application Deadline

**Online:**
- June 23, 2027 – Final Application Deadline

Sources: 
https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/

Retrieved chunks:
  [1] Events & Deadlines -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/
  [2] Events & Deadlines -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/
  [3] How to Apply -- https://d

Q: What is the best pizza place in Chicago?

A: I don't have that information in the MS in Applied Data Science website content.

Retrieved chunks:
  [1] Explore the MS-ADS Campus -- https://datascience.uchicago.edu/explore-the-ms-ads-campus/
  [2] Donald Patchell, MSE, MBA -- https://datascience.uchicago.edu/people/donald-patchell-mse-mba/
  [3] Faculty, Instructors, Staff -- https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/instructors-staff/
  [4] In-Person Program -- https://datascience.uchicago.edu/education/masters-programs/in-person-program/


## Baseline vs. heading-aware: side-by-side
`answered` = the model gave an answer rather than the fallback message (for the last, out-of-scope question the
fallback is the *correct* behavior).

In [10]:
import pandas as pd

pd.set_option("display.max_colwidth", None)


def answered(result):
    return FALLBACK not in result["answer"]


comparison = pd.DataFrame({
    "question": TEST_QUESTIONS,
    "baseline answered": [answered(r) for r in baseline_results],
    "headings answered": [answered(r) for r in headings_results],
    "baseline answer": [r["answer"] for r in baseline_results],
    "headings answer": [r["answer"] for r in headings_results],
})
comparison.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap", "vertical-align": "top"})

,question,baseline answered,headings answered,baseline answer,headings answer
0,What are the core courses in the MS in Applied Data Science program?,False,True,I don't have that information in the MS in Applied Data Science website content.,The core courses in the MS in Applied Data Science program are: 1. Time Series Analysis and Forecasting 2. Statistical Models for Data Science 3. Machine Learning I 4. Machine Learning II 5. Cloud-Native Data Engineering 6. Leadership and Consulting for Data Science 7. Data Science Capstone Project Sources: https://datascience.uchicago.edu/education/masters-programs/online-program/
1,What are the admission requirements for the MS in Applied Data Science program?,True,True,"The admission requirements for the MS in Applied Data Science program include some prior experience with Python or R. Additionally, applicants must submit a completed application, which will be reviewed by the committee. The application portal may close early if the cohort is filled. Sources: - https://datascience.uchicago.edu/news/applying-to-the-ms-in-applied-data-science-program-heres-what-we-look-for/ - https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/",The admission requirements for the MS in Applied Data Science program include: - Next Application Deadlines - Letters of Recommendation - Candidate Statement - Resume/CV - Programming Supplement - Virtual Portfolio - Transcripts from all previous colleges and universities attended - Application Fee - English Language Requirement (for international students) Sources: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/
2,Can you provide information about the capstone project?,True,True,"The Capstone Project in the MS in Applied Data Science program is a required component that is completed over two quarters. It involves research design, implementation, and writing. Full-time students typically start their capstone project in their third quarter, while part-time students generally begin in their fifth quarter. In the second quarter, the capstone class focuses on providing students with maximum flexibility to work with their Capstone advisors on concluding the execution of the analytic methodology and any client/sponsor deliverables. It also supports students in curating and delivering key project communications, including a formal research paper, a formal business presentation, and a live presentation during the Capstone Showcase with a Q&A session. Sources: - https://datascience.uchicago.edu/education/masters-programs/in-person-program/ - https://datascience.uchicago.edu/education/masters-programs/online-program/","The capstone project in the MS in Applied Data Science program allows students to work on real-world problems for actual clients, enhancing their collaboration skills and providing networking opportunities. Faculty-mentored teams contribute to various sectors, including finance, entertainment, and automotive. In the second quarter, students focus on executing their analytic methodology and preparing deliverables, which include a formal research paper, a business presentation, and a live presentation during the Capstone Showcase. Sources: - https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/ - https://datascience.uchicago.edu/education/masters-programs/in-person-program/ - https://datascience.uchicago.edu/education/masters-programs/online-program/"
3,How much is tuition for the program?,True,True,"Tuition for the MS in Applied Data Science program is $6,384 per course, totaling $76,608 for 12 courses or $114,912 for 18 courses. Additionally, there is a non-refundable program enrollment deposit of $1,500, which is credited toward your first quarter’s tuition balance. Sources: https://datascience.uchicago.edu/education/tuition-fees-aid/","Tuition for the MS in Applied Data Science program is $6,384 per course, t

---
# Evaluation with RAGAS
Eight hand-picked questions show *whether* something changed; RAGAS measures *how much*, on a fixed test set, so
approaches built on different branches can be compared fairly.

**Test set** — [`eval/testset.json`](eval/testset.json): 15 in-scope questions (the 3 from the project brief plus
admissions, tuition, deadlines, visas, curriculum, and logistics) with **hand-written reference answers** taken from the
scraped site, the pages each answer comes from, and 2 out-of-scope questions.

**Metrics** — computed by [`rag_eval.py`](rag_eval.py), averaged over in-scope questions (all 0–1, higher is better):

| Metric | Stage | Question it answers |
|---|---|---|
| `context_precision` | retrieval | Are the chunks that support the reference answer ranked at the top? |
| `context_recall` | retrieval | Do the retrieved chunks contain everything the reference answer says? |
| `source_hit` | retrieval | Did at least one retrieved chunk come from a page the answer is on? (no LLM) |
| `faithfulness` | generation | Is every claim in the answer supported by the retrieved chunks? (hallucination check) |
| `answer_relevancy` | generation | Does the answer address the question that was asked? |
| `factual_correctness` | end-to-end | Claim-level F1 of the answer vs. the reference answer |
| `answered` | end-to-end | Share of in-scope questions answered instead of returning the fallback |
| `out_of_scope_refusal` | guardrail | Share of out-of-scope questions that correctly got the fallback |

**Notes**
- The judge is **`gpt-4o`**, deliberately a different (stronger) model than the `gpt-4o-mini` generator, to limit a model
  grading its own answers. Keep `JUDGE_MODEL` unchanged across branches or scores stop being comparable.
- The trailing `Sources:` line is stripped before judging, so citations don't count as claims.
- When a pipeline returns the fallback for an in-scope question, `faithfulness` has no claims to check (left blank);
  that miss shows up in `answered`, `answer_relevancy`, and `factual_correctness` instead.
- Cost: on the order of $1 per pipeline in judge calls (the actual cost is printed after each run). LLM judges are not perfectly deterministic, so treat
  differences of a few points as noise.

In [ ]:
import rag_eval

testset = rag_eval.load_testset()
print(f"{len(testset)} test questions ({sum(t['in_scope'] for t in testset)} in scope)")

RUN_CONFIG = {"chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP, "top_k": TOP_K,
              "embed_model": EMBED_MODEL, "llm_model": LLM_MODEL}
PIPELINES = {  # run name -> (retriever, what differs in this run)
    "baseline": (retriever, {"chunking": "recursive_character"}),
    "heading_aware": (retriever_v2, {"chunking": "markdown_headers + breadcrumb + outline"}),
}

eval_scores = {}
for run_name, (run_retriever, run_config) in PIPELINES.items():
    print(f"\n=== {run_name} ===")
    runs = rag_eval.collect(lambda q: ask(q, retriever=run_retriever, verbose=False), testset)
    eval_scores[run_name] = rag_eval.score(runs, fallback=FALLBACK)
    rag_eval.save_results(eval_scores[run_name], run_name, config={**RUN_CONFIG, **run_config})

## Leaderboard
Every run saved in `eval/results/` — after merging teammates' branches, their runs show up here too.

In [ ]:
rag_eval.leaderboard().drop(columns="evaluated_at").style.format(precision=2).background_gradient(
    cmap="RdYlGn", vmin=0, vmax=1)

## Per-question breakdown
Where each pipeline wins or loses (blank = not applicable, e.g. faithfulness when the pipeline returned the fallback).

In [ ]:
PER_QUESTION = ["context_recall", "faithfulness", "factual_correctness"]
per_question = pd.concat({name: s[s["in_scope"]].set_index("id")[PER_QUESTION] for name, s in eval_scores.items()},
                         axis=1).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)
per_question.style.format(precision=2, na_rep="").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)

## Comparing a new approach (for teammates)
1. Branch off `main` and change the pipeline (chunking, retriever, reranker, prompt, model, ...).
2. Add it to `PIPELINES` with a **unique run name** (e.g. `"rerank_cohere"`) and a short description of what changed.
3. Run the evaluation cell; commit the new `eval/results/<run_name>.json` and `.csv` with your code.
4. **Don't edit** `eval/testset.json` or the judge settings in `rag_eval.py` on a feature branch — change them on `main`
   together and re-score every approach, otherwise the numbers aren't comparable.

## Next steps
- **Retrieval:** hybrid search (BM25 + dense), reranking (e.g., Cohere / cross-encoder), metadata filters
- **Evaluation:** grow the test set (multi-hop, paraphrased, and adversarial questions); track scores per question type
- **Responsible AI:** PII redaction (faculty contact info), input guardrails; use `faithfulness` as a runtime
  groundedness check
- **UI:** Streamlit chat interface showing the answer alongside the retrieved sources